# QLoRA for EGAIS assistant (Colab T4)

This notebook trains a LoRA adapter from `data/lora/train.jsonl` generated by `scripts/build_lora_dataset.py`.

In [ ]:
!pip install -q -U pip
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Put train/eval files to Drive and update paths:
TRAIN_PATH = '/content/drive/MyDrive/egais-lora/train.jsonl'
EVAL_PATH = '/content/drive/MyDrive/egais-lora/eval.jsonl'
OUT_DIR = '/content/drive/MyDrive/egais-lora/adapter-qwen25-05b'

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': TRAIN_PATH, 'eval': EVAL_PATH})
dataset

In [ ]:
# For free Colab T4, start with 0.5B. Switch to 1.5B only if memory allows.
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

# T4-safe defaults
MAX_SEQ_LEN = 768
LR = 2e-4
EPOCHS = 2
BATCH = 1
GRAD_ACC = 16
USE_EVAL = False

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)

def format_example(ex):
    msgs = ex['messages']
    text = ''
    for m in msgs:
        text += f"<{m['role']}>\n{m['content']}\n"
    return {'text': text}

train_ds = dataset['train'].map(format_example)
eval_ds = dataset['eval'].map(format_example) if USE_EVAL and 'eval' in dataset else None

# transformers API differs by version: some use `evaluation_strategy`, older/newer may use `eval_strategy`.
import inspect
train_args_params = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs = {
    'output_dir': '/content/out',
    'per_device_train_batch_size': BATCH,
    'gradient_accumulation_steps': GRAD_ACC,
    'learning_rate': LR,
    'num_train_epochs': EPOCHS,
    'logging_steps': 20,
    'save_strategy': 'epoch',
    # Stable precision mode for mixed-version Colab stacks.
    'bf16': False,
    'fp16': False,
    'optim': 'paged_adamw_8bit',
    'group_by_length': True,
    'report_to': 'none',
}
if 'evaluation_strategy' in train_args_params:
    training_kwargs['evaluation_strategy'] = 'epoch' if eval_ds is not None else 'no'
elif 'eval_strategy' in train_args_params:
    training_kwargs['eval_strategy'] = 'epoch' if eval_ds is not None else 'no'

args = TrainingArguments(**training_kwargs)

# trl API differs by version: some use `tokenizer`, newer use `processing_class`.
sft_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'train_dataset': train_ds,
    'eval_dataset': eval_ds,
    'peft_config': peft_config,
    'args': args,
}
if 'tokenizer' in sft_params:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in sft_params:
    trainer_kwargs['processing_class'] = tokenizer
if 'dataset_text_field' in sft_params:
    trainer_kwargs['dataset_text_field'] = 'text'
if 'max_seq_length' in sft_params:
    trainer_kwargs['max_seq_length'] = MAX_SEQ_LEN

def make_trainer(max_seq_len: int):
    local_kwargs = dict(trainer_kwargs)
    if 'max_seq_length' in sft_params:
        local_kwargs['max_seq_length'] = max_seq_len
    return SFTTrainer(**local_kwargs)

seq_schedule = [MAX_SEQ_LEN, 640, 512]
last_err = None
trainer = None

for seq_len in seq_schedule:
    try:
        print(f'Trying training with max_seq_length={seq_len}')
        torch.cuda.empty_cache()
        trainer = make_trainer(seq_len)
        trainer.train()
        last_err = None
        break
    except torch.cuda.OutOfMemoryError as e:
        last_err = e
        print(f'OOM at max_seq_length={seq_len}. Retrying with smaller context...')
        torch.cuda.empty_cache()

if last_err is not None:
    raise last_err

trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Saved adapter to', OUT_DIR)

In [ ]:
# Quick sanity check in Colab
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, OUT_DIR)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

prompt = 'Вопрос: какие документы нужны для лицензии на перевозку этилового спирта?\nОтвет:'
inputs = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))